# DemandLens: model development and evaluation

The forecasting approach step by step: features, leakage check, rolling-origin backtest against baselines, calibration of the 90th percentile, why a separate mean forecast is needed, feature importance, and the anomaly detector's evaluation. It calls the same functions as the application's analysis run.

**Run time:** the backtest trains 18 models on up to about 700,000 rows each; allow 5-15 minutes on a laptop. Fold results are cached in `data/cache`, so re-running is fast.

In [ ]:
import os, sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=RuntimeWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from analytics.config import AnalysisConfig
from analytics.ingest import read_online_retail, clean_online_retail
from analytics.preprocess import build_panel

plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": .3})
GREEN, INK, AMBER, BRICK = "#1E6B5C", "#1B2B3A", "#B7791F", "#A63D2F"

# The UCI Online Retail II file (CSV or the original .xlsx). Set DEMANDLENS_DATA to use another file,
# for example the synthetic file from scripts/generate_synthetic.py.
candidates = [os.environ.get("DEMANDLENS_DATA"), ROOT / "data/online_retail_II.csv", ROOT / "data/online_retail_II.xlsx"]
DATA = next(Path(p) for p in candidates if p and Path(p).exists())
print("Using", DATA)

In [ ]:
raw = read_online_retail(DATA)
movements, products, report = clean_online_retail(raw)
movements["product_id"] = pd.factorize(movements["sku"])[0] + 1     # stand-in ids (the app uses database ids)
sku_of = movements.drop_duplicates("product_id").set_index("product_id")["sku"]
name_of = products.set_index("sku")["name"]
panel = build_panel(movements)
cfg = AnalysisConfig()
T = len(panel.weeks)
print(f"{len(movements):,} ledger movements, {panel.units.shape[0]:,} products, {T} complete weeks "
      f"({panel.weeks[0].date()} to {panel.last_week.date()})")

## 1. Features

One row per (product, forecast origin week *t*, horizon *h*). Features use data up to week *t* only; the target is demand in week *t+h*.

In [ ]:
from analytics.features import DesignBuilder, FEATURES, add_profile
from analytics.segmentation import profile
builder = DesignBuilder(panel, cfg, categories=None, holiday_country="GB")
rows = builder.rows(0, T - 1, require_target=False)
print(f"{len(rows):,} design rows, {len(FEATURES)} features:", ", ".join(FEATURES))
display(add_profile(rows[rows["_t"] == T - 5].head(5), profile(panel, T - 5, cfg), panel)[FEATURES + ["_y"]])

### Leakage check
Change every value after week *t*: the features at *t* must not change (the same test runs in `tests/test_models.py`).

In [ ]:
t = T - 10
before = DesignBuilder(panel, cfg).rows(t, t, require_target=True)
saved = panel.units.copy(); panel.units[:, t + 1:] = 9999
after = DesignBuilder(panel, cfg).rows(t, t, require_target=True); panel.units[:] = saved
cols = [c for c in before.columns if not c.startswith("_") and c != "y_last_year"]
print("Features unchanged by future data:", before[cols].equals(after[cols]))

## 2. Rolling-origin backtest

Six forecast origins, four weeks apart, across the last 24 weeks. At each origin every method sees only earlier data.

In [ ]:
import time
from analytics.evaluate import run_backtest, summarise, origins_for
print("Origins:", [str(panel.weeks[o].date()) for o in origins_for(panel, cfg)])
t0 = time.time()
bt = run_backtest(panel, builder, rows, cfg, progress=print, cache_dir=ROOT / "data/cache", fingerprint=f"nb_{DATA.stem}_{T}_" + __import__("hashlib").sha1(repr(cfg).encode()).hexdigest()[:10])
patterns = profile(panel, T - 1, cfg).set_index("product_id")["demand_pattern"]
summary = summarise(bt, patterns)
print(f"{len(bt):,} forecast points in {time.time() - t0:.0f} s")

In [ ]:
overall = pd.DataFrame(summary["overall"]).T[["mae", "wape", "mase", "bias", "n"]]
display(overall.sort_values("mase").style.format({"mae": "{:.2f}", "wape": "{:.3f}", "mase": "{:.3f}", "bias": "{:.2f}"})
        .highlight_min(subset=["mae", "wape", "mase"], color="#DCEBE6"))

In [ ]:
pd.DataFrame(summary["by_horizon"]).T.plot(marker="o", title="MASE by weeks ahead"); plt.ylabel("MASE"); plt.show()
bp = pd.DataFrame({p: {m: v[m]["mase"] for m in overall.index if m in v} for p, v in summary["by_pattern"].items()}).T
bp["products"] = [summary["by_pattern"][p]["products"] for p in bp.index]
bp["chosen"] = [summary["selection"][p]["method"] for p in bp.index]
display(bp.round(3))

### Is the improvement consistent?
A paired comparison per product: how often does the model beat the best simple method?

In [ ]:
best_base = min((m for m in overall.index if m != "gbm"), key=lambda m: overall.loc[m, "mase"])
g = bt[bt["gbm"].notna() & (bt["scale"] > 0)]
per = g.assign(e_gbm=(g["actual"] - g["gbm"]).abs() / g["scale"], e_base=(g["actual"] - g[best_base]).abs() / g["scale"]) \
       .groupby("product_id")[["e_gbm", "e_base"]].mean()
print(f"Model more accurate than {best_base} for {(per.e_gbm < per.e_base).mean():.1%} of {len(per):,} products")
from scipy.stats import wilcoxon
print("Wilcoxon signed-rank test on per-product MASE:", wilcoxon(per["e_gbm"], per["e_base"]))

## 3. Uncertainty and the mean forecast

The P90 should cover about 90% of weeks. The median forecast should not be used for totals: for skewed demand the sum of weekly medians falls well short of total demand.

In [ ]:
cal = summary["calibration"]
print(f"P90 coverage: {cal['gbm_p90_coverage']:.1%}   pinball loss: {cal['gbm_p90_pinball']:.2f}")
display(pd.Series(cal["by_pattern"], name="P90 coverage").round(3))
g = bt[bt["gbm"].notna()]
tot = pd.Series({"actual": g["actual"].sum(), "sum of weekly medians (P50)": g["gbm"].sum(), "sum of mean forecasts": g["gbm_mean"].sum()})
(tot / tot["actual"]).plot.bar(color=[INK, AMBER, GREEN], title="Total forecast relative to actual demand"); plt.axhline(1, color="grey", ls="--"); plt.show()

## 4. What drives the forecasts

Permutation importance of the median model on the final origin: how much the error grows when one feature is shuffled.

In [ ]:
from sklearn.inspection import permutation_importance
from analytics.evaluate import training_rows
from analytics.forecasting import QuantileGBM
o = origins_for(panel, cfg)[-1]
prof_o = profile(panel, o, cfg)
tr = training_rows(rows, o, cfg)
model = QuantileGBM(cfg.gbm_params, cfg.random_state, with_mean=False).fit(add_profile(tr, prof_o, panel), tr["_y"].to_numpy())
test = rows[(rows["_t"] == o) & (rows["age"] >= cfg.min_history_weeks) & rows["_y"].notna()]
Xt = add_profile(test, prof_o, panel)[model.features].to_numpy(np.float32)
if model.dropped: print("Not used (no variation in the training data):", ", ".join(model.dropped))
yt = np.log1p(test["_y"].to_numpy())
imp = permutation_importance(model.models[0.5], Xt, yt, n_repeats=3, random_state=0, scoring="neg_mean_absolute_error")
pd.Series(imp.importances_mean, index=model.features).sort_values().plot.barh(color=GREEN, figsize=(8, 7), title="Permutation importance (P50 model)"); plt.show()

## 5. Example forecasts

In [ ]:
top = profile(panel, T - 1, cfg).nlargest(4, "revenue_52w")["product_id"]
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, pid in zip(axes.ravel(), top):
    i = panel.row(pid); b = bt[bt["product_id"] == pid].sort_values("target")
    ax.plot(panel.weeks[-40:], panel.units[i, -40:], color=INK, label="actual")
    ax.plot(panel.weeks[b["target"]], b["gbm"], ".", color=GREEN, label="P50 (at the time)")
    ax.fill_between(panel.weeks[b["target"]], 0, b["gbm_p90"], color=GREEN, alpha=.15, label="up to P90")
    ax.set_title(str(name_of.get(sku_of[pid], pid))[:40], fontsize=10)
axes[0, 0].legend(); plt.tight_layout(); plt.show()

## 6. Unusual-sales detector

Anomalies of known size are planted in a copy of the backtest weeks (see `analytics/anomalies.py`).

In [ ]:
from analytics.anomalies import injection_evaluation
from analytics.baselines import METHODS
sel = summary["selection"]
pts = bt.assign(pattern=bt["product_id"].map(patterns).fillna("insufficient"))
pts = pts.assign(p50=np.where(pts["gbm"].notna(), pts["gbm"], pts["moving_average"]),
                 p90=np.where(pts["gbm_p90"].notna(), pts["gbm_p90"], pts["moving_average"] * 2),
                 week_start=panel.weeks[pts["target"]])
ev = injection_evaluation(pts[["product_id", "week_start", "actual", "p50", "p90", "pattern"]], cfg)
print(f"Weeks flagged without planting: {ev['flag_rate']:.1%}")
display(pd.DataFrame(ev["spikes"]).T.round(3)); print("Zero-sales weeks caught:", ev["drops"])
display(pd.DataFrame(ev["threshold_sensitivity"]).round(3))

## 7. Optional: tuning

The `flask --app run tune --user <you>` command runs a small grid search with validation origins *before* the backtest window, so the backtest above stays an untouched test. Expect about 20-40 minutes on the full dataset.